<left>
    <img src="https://weclouddata.s3.amazonaws.com/images/logos/wcd_logo_new_2.png" width='20%'>
</left>

<h1 align="left"> Building Minimal Tool Agent </h1>
<center align="left"> <font size='4'>  Developed by: </font><font size='4' color='#33AAFBD'>WeCloudData</font></center>
<br>

**A simple example showing how to build a basic tool-using agent.**

What you will learn:
- How to safely run math expressions
- What a "Tool" is and how to store them
- How an agent decides which tool to use
- How the agent loops, calls tools, and stops with a final answer

## How the Agent Works

It's simpler than it sounds:

1. You give the agent a question  
2. The agent looks at the question and decides: "Which tool should I use?"  
3. It calls that tool and gets some output  
4. Then it asks itself: "Should I stop now and give the final answer, or do I need another tool?"  
5. It keeps going until it decides the answer is ready.

We are using simple rules (called heuristics) to make decisions. Later you can replace these rules with a real LLM if you want.

## 1. Example: Safe Arithmetic Evaluator

We need a safe way to calculate math.  Python's `eval()`is dangerous because someone could trick it into deleting files or worse.

Instead, we'll use Python's `ast` module to carefully read the math expression and calculate it step by step.

This is a very common and safe pattern used in real projects.

In [ ]:
import ast
import operator as op
from dataclasses import dataclass
from typing import Callable, Dict, Any, Tuple

# Mapping of Python operators to real functions
_operators = {
    ast.Add: op.add,
    ast.Sub: op.sub,
    ast.Mult: op.mul,
    ast.Div: op.truediv,
    ast.Pow: op.pow,
    ast.USub: op.neg,
}

def safe_eval(expr: str) -> float:
    """Safely calculate simple math expressions """
    try:
        node = ast.parse(expr, mode='eval')
    except Exception as e:
        raise ValueError(f"Invalid expression: {expr}") from e

    def _eval(n):
        if isinstance(n, ast.Expression):
            return _eval(n.body)
        if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)):
            return n.value
        if isinstance(n, ast.BinOp):
            left = _eval(n.left)
            right = _eval(n.right)
            op_type = type(n.op)
            if op_type in _operators:
                return _operators[op_type](left, right)
            raise ValueError(f"Unsupported operator: {op_type}")
        if isinstance(n, ast.UnaryOp):
            operand = _eval(n.operand)
            op_type = type(n.op)
            if op_type in _operators:
                return _operators[op_type](operand)
            raise ValueError(f"Unsupported unary operator: {op_type}")
        raise ValueError(f"Unsupported expression: {ast.dump(n)}")

    return _eval(node)

# Test it
print(safe_eval("2 + 10 * 4"))
print(safe_eval("2 ** 9"))

42
512


## 2. Tool Class + Registry

If we just write a bunch of functions, the code quickly becomes messy. The agent wouldn't know what tools exist, what they do, or when to stop.

So two things are created:

### The Tool Class
It's like an **ID card** for every tool. Every tool gets:
- A name
- The actual function it runs
- A short description
- A flag saying whether this tool gives the final answer

Using a class makes everything consistent and easy to manage.

### The ToolRegistry
This is like a **toolbox** or a library catalog.  
Instead of throwing all tools in different places, we keep them together in one clean place.

Benefits:
- Easy to add new tools later
- The agent can easily look up any tool by name
- Clean separation: tools do the work, agent makes decisions
- This exact pattern is used in professional agent frameworks (LangChain, etc.)

In [ ]:
#TOOL CLASS
@dataclass
class Tool:
    name: str
    func: Callable[[str], Dict[str, Any]]
    description: str = ""
    returns_final: bool = False

#TOOL REGISTRY

class ToolRegistry:
    def __init__(self):
        self._tools: Dict[str, Tool] = {}

    def register(self, tool: Tool):
        self._tools[tool.name] = tool

    def get(self, name: str) -> Tool:
        return self._tools[name]

#Create registry
registry = ToolRegistry()
print("ToolRegistry created!")

ToolRegistry created!


## 3. Creating Tools

Now define the actual tools:

- **calculator** → does safe math
- **search** → looks up information in our tiny knowledge base
- **summarize** → shortens text or search results
- **answer** → formats the final answer and tells the agent to stop

After defining them, we register each one into the registry.

In [ ]:
# Tiny knowledge base
KB = {
    "python": "Python is a popular programming language used for scripting, data science, and web development.",
    "agent": "An agent is a program that takes actions to achieve goals; tool agents call tools to help.",
    "jupiter": "Jupiter is the largest planet in the Solar System.",
}

def calculator_tool(query: str) -> Dict[str, Any]:
    try:
        result = safe_eval(query)
        return {"output": str(result), "tool": "calculator", "final": False}
    except Exception as e:
        return {"output": f"Calculator error: {e}", "tool": "calculator", "final": False}

def search_tool(query: str) -> Dict[str, Any]:
    q = query.lower()
    hits = []
    for k, v in KB.items():
        if k in q or any(word in q for word in k.split()):
            hits.append({"key": k, "text": v})
    if not hits:
        for k, v in KB.items():
            if any(tok in q for tok in k.split()):
                hits.append({"key": k, "text": v})
    return {
        "output": hits if hits else "No results found.",
        "tool": "search",
        "final": False
    }

def summarize_tool(query: str) -> Dict[str, Any]:
    if isinstance(query, list):
        summary = "; ".join(f"{h['key']}: {h['text'][:80]}" for h in query)
        return {"output": summary, "tool": "summarize", "final": False}
    short = (query[:200] + "...") if len(query) > 200 else query
    return {"output": f"Summary: {short}", "tool": "summarize", "final": False}

def answer_tool(query: str) -> Dict[str, Any]:
    return {"output": f"Final answer: {query}", "tool": "answer", "final": True}

# Register all tools
registry.register(Tool("calculator", calculator_tool, "Safe arithmetic evaluator"))
registry.register(Tool("search", search_tool, "Simple keyword search over KB"))
registry.register(Tool("summarize", summarize_tool, "Summarize search results"))
registry.register(Tool("answer", answer_tool, "Produce final answer", returns_final=True))

print("All tools registered successfully!")
print("Available tools:", list(registry._tools.keys()))

All tools registered successfully!
Available tools: ['calculator', 'search', 'summarize', 'answer']


## 4. The Agent – The Brain

Now comes the interesting part: the agent itself.

We have two functions:
- `choose_tool()` → decides which tool to use based on simple rules
- `run_agent()` → runs the loop, calls tools, and stops when ready

The decision logic here is smart enough so it doesn't repeat the same tool forever.

In [ ]:
import re

def choose_tool(current_state):
    text = str(current_state).strip().lower()

    # 1. Math
    if re.search(r"^[0-9\s\+\-\*/\.\*\*]+$", text.replace(" ", "")):
        return "calculator", "looks like math"

    # 2. If we already summarized (text starts with "summary:")
    if text.startswith("summary:"):
        return "answer", "already summarized → final answer"

    # 3. If we have search results (list)
    if isinstance(current_state, list):
        return "summarize", "we have search hits → summarize them"

    # 4. User asked to summarize
    if any(word in text for word in ["summarize", "short", "tl;dr"]):
        return "summarize", "user wants summary"

    # 5. Information request or topic
    if any(word in text for word in ["what is", "who is", "tell me about", "define", "explain"]):
        return "search", "information request"

    for key in KB:
        if key in text:
            return "search", f"topic: {key}"

    # 6. Default final answer
    return "answer", "default final answer"


def run_agent(initial_query: str, max_steps: int = 6):
    state = initial_query
    print(f" Agent started with: {initial_query}\n")

    for step in range(1, max_steps + 1):
        tool_name, reason = choose_tool(state)
        tool = registry.get(tool_name)

        print(f"Step {step}: {tool_name} → {reason}")

        # Call tool
        if tool_name == "summarize" and isinstance(state, list):
            result = tool.func(state)
        else:
            result = tool.func(str(state))

        print(f"   → {str(result['output'])[:120]}...\n")

        # Stop condition
        if result.get("final") or tool.returns_final:
            print(" Done! Final answer ready.\n")
            return result

        state = result["output"]

    print("Max steps reached → forcing final answer")
    return registry.get("answer").func(str(state))


print(" Agent loaded")

 Agent loaded


Test the Agent

In [ ]:
examples = [
    "2 + 3 * 4",
    "What is Python?",
    "Tell me about Jupiter",
    "Python and AI agent ",
    "Explain the agent concept"
]

for q in examples:
    print("\n" + "="*60)
    print(f"QUERY: {q}")
    result = run_agent(q)
    print("FINAL:", result["output"])
    print("="*60)


QUERY: 2 + 3 * 4
 Agent started with: 2 + 3 * 4

Step 1: calculator → looks like math
   → 14...

Step 2: calculator → looks like math
   → 14...

Step 3: calculator → looks like math
   → 14...

Step 4: calculator → looks like math
   → 14...

Step 5: calculator → looks like math
   → 14...

Step 6: calculator → looks like math
   → 14...

Max steps reached → forcing final answer
FINAL: Final answer: 14

QUERY: What is Python?
 Agent started with: What is Python?

Step 1: search → information request
   → [{'key': 'python', 'text': 'Python is a popular programming language used for scripting, data science, and web developme...

Step 2: summarize → we have search hits → summarize them
   → python: Python is a popular programming language used for scripting, data science, and w...

Step 3: search → topic: python
   → [{'key': 'python', 'text': 'Python is a popular programming language used for scripting, data science, and web developme...

Step 4: summarize → we have search hits → summ

In [ ]:
while True:
    q = input("\nYour question (press Enter to quit): ").strip()
    if not q:
        print("Goodbye!")
        break
    result = run_agent(q)
    print("FINAL RESULT →", result["output"])


Your question (press Enter to quit): python
 Agent started with: python

Step 1: search → topic: python
   → [{'key': 'python', 'text': 'Python is a popular programming language used for scripting, data science, and web developme...

Step 2: summarize → we have search hits → summarize them
   → python: Python is a popular programming language used for scripting, data science, and w...

Step 3: search → topic: python
   → [{'key': 'python', 'text': 'Python is a popular programming language used for scripting, data science, and web developme...

Step 4: summarize → we have search hits → summarize them
   → python: Python is a popular programming language used for scripting, data science, and w...

Step 5: search → topic: python
   → [{'key': 'python', 'text': 'Python is a popular programming language used for scripting, data science, and web developme...

Step 6: summarize → we have search hits → summarize them
   → python: Python is a popular programming language used for scripting, da